In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/loan_clean.csv")                 # 0/1 version, for the model
labeled = pd.read_csv("../data/loan_clean_labeled.csv")    # text version, for the dropdown options
print(df.shape, labeled.shape)    # expect (20000, 39) (20000, 28)

(20000, 39) (20000, 28)


#### rebuild v3

In [2]:
DROP_V3 = [
    "Experience",                                    # repeats Age (Step 4, Exercise 5)
    "NumberOfDependents", "NumberOfOpenCreditLines", "NumberOfCreditInquiries",
    "UtilityBillsPaymentHistory", "JobTenure",
    "SavingsAccountBalance", "CheckingAccountBalance",
    "MaritalStatus_Married", "MaritalStatus_Single", "HomeOwnershipStatus_Own",
    "LoanPurpose_Education", "LoanPurpose_Home",
]                                                    # 13 names = Experience + the 12 weak features

X = df.drop(columns="RiskScore")
y = df["RiskScore"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_v3 = X_train.drop(columns=DROP_V3)
X_test_v3 = X_test.drop(columns=DROP_V3)
assert X_train_v3.shape[1] == 25

model = LinearRegression().fit(X_train_v3, y_train)
pred = model.predict(X_test_v3)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
print(round(r2, 3), round(rmse, 3), round(mae, 3))    # expect 0.759 3.871 2.964

0.759 3.871 2.964


#### Save the model and its metadata

In [3]:
Path("../models").mkdir(exist_ok=True)
joblib.dump(model, "../models/risk_model_v3.joblib")

features = X_train_v3.columns.tolist()
cat_cols = labeled.select_dtypes(exclude="number").columns.tolist()    # the 5 text columns

# A text column is still an input if at least one of its 0/1 columns survived in v3
cat_used = [c for c in cat_cols if any(f.startswith(c + "_") for f in features)]
# Everything that is not a 0/1 column from a text column is a plain number input
num_used = [f for f in features if not any(f.startswith(c + "_") for c in cat_cols)]
print(len(num_used), cat_used)    # expect 14 and the 5 text columns

14 ['EmploymentStatus', 'EducationLevel', 'MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose']


In [4]:
meta = {
    "model_version": "v3",
    "sklearn_version": sklearn.__version__,
    "features": features,                      # the exact 25 columns, in order
    "numeric": {
        c: {
            "min": float(X_train_v3[c].min()),
            "median": float(X_train_v3[c].median()),
            "max": float(X_train_v3[c].max()),
            "is_int": bool(pd.api.types.is_integer_dtype(X_train_v3[c])),
        }
        for c in num_used
    },
    "categorical": {
        c: {"options": sorted(labeled[c].unique().tolist()), "default": labeled[c].mode()[0]}
        for c in cat_used
    },
    "means": {f: float(X_train_v3[f].mean()) for f in features},
    "target": {
        "min": float(y_train.min()),
        "max": float(y_train.max()),
        "percentiles": np.percentile(y_train, range(101)).round(3).tolist(),
    },
    "metrics_test": {"r2": round(r2, 3), "rmse": round(rmse, 3), "mae": round(mae, 3)},
}
Path("../models/model_meta.json").write_text(json.dumps(meta, indent=2))
print(meta["sklearn_version"], len(meta["features"]), meta["target"]["min"], meta["target"]["max"])

1.9.1 25 28.8 84.0


In [5]:
reloaded = joblib.load("../models/risk_model_v3.joblib")
print(np.allclose(reloaded.predict(X_test_v3), pred))    # expect True

True


In [6]:
sys.path.insert(0, "..")    # lets the notebook import predict.py from the repo root
from predict import META, check_inputs, encode, explain, percentile_rank, predict

In [7]:
test_idx = X_test_v3.index[:500]
needed = list(META["numeric"]) + list(META["categorical"])

rows = labeled.loc[test_idx, needed].to_dict("records")
mine = np.array([predict(r) for r in rows])
expected = np.clip(model.predict(X_test_v3.loc[test_idx]), META["target"]["min"], META["target"]["max"])

print(np.abs(mine - expected).max())    # expect 0.0 (or something like 1e-12)

1.4210854715202004e-14


In [8]:
typical = {c: m["median"] for c, m in META["numeric"].items()}
typical.update({c: info["default"] for c, info in META["categorical"].items()})

base = predict(typical)
print("typical borrower:  ", round(base, 1))
print("add bankruptcy:    ", round(predict({**typical, "BankruptcyHistory": 1}) - base, 1))      # close to +13.2
print("add past default:  ", round(predict({**typical, "PreviousLoanDefaults": 1}) - base, 1))  # close to +6.9

typical borrower:   49.1
add bankruptcy:     13.2
add past default:   6.9


In [9]:
print(check_inputs({**typical, "CreditScore": 900}))    # one warning about CreditScore

try:
    predict({**typical, "EmploymentStatus": "Retired"})
except ValueError as e:
    print("caught:", e)                                 # a typo is stopped, not silently scored

['CreditScore = 900 is outside the training range (343 to 712), so treat the score with care.']
caught: EmploymentStatus: 'Retired' is not one of ['Employed', 'Self-Employed', 'Unemployed']
